# Business Insights for Power BI Dashboard
This notebook prepares the cleaned dataset and summary tables needed for the Power BI dashboard. It creates the dashboard-ready fields and saves exports that support KPI cards, risk analysis, and financial behavior visuals.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Absolute path to project root
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "Notebooks" else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", os.getcwd())

In [ ]:
# Load dataset with low_memory=False for type consistency
DATASET_PATH = PROJECT_ROOT / "Dataset" / "dataset.csv"
df = pd.read_csv(DATASET_PATH, low_memory=False)

# Ensure numeric interest rate
if df["int_rate"].dtype == object:
    df["int_rate"] = df["int_rate"].astype(str).str.replace("%", "", regex=False)
    df["int_rate"] = pd.to_numeric(df["int_rate"], errors="coerce")

# Derived dashboard fields
if "is_default" not in df.columns:
    df["is_default"] = df["loan_status"].isin(["Charged Off", "Default"]).astype(int)

if "fico_avg" not in df.columns:
    df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

if "loan_term_numeric" not in df.columns:
    df["loan_term_numeric"] = pd.to_numeric(df["term"].str.extract(r"(\d+)")[0], errors="coerce")

# Income bracket for dashboard segmentation
income_bins = [0, 30000, 60000, 90000, 120000, 150000, np.inf]
income_labels = ["0-30K", "30K-60K", "60K-90K", "90K-120K", "120K-150K", ">150K"]
df["income_bracket"] = pd.cut(df["annual_inc"], bins=income_bins, labels=income_labels)

# Default category label
if "default_rate_category" not in df.columns:
    df["default_rate_category"] = np.where(df["is_default"] == 1, "Bad Loan", "Good Loan")

print("Loaded rows:", len(df))
print("Sample fields:", ["loan_amnt", "term", "int_rate", "grade", "purpose", "is_default", "fico_avg"])

In [ ]:
# Save cleaned dataset for Power BI
CLEAN_PATH = PROJECT_ROOT / "Dataset" / "dashboard_data.csv"
selected_cols = [
    "loan_amnt", "term", "loan_term_numeric", "int_rate", "installment",
    "grade", "sub_grade", "purpose", "emp_length", "home_ownership",
    "annual_inc", "annual_inc", "verification_status", "dti", "delinq_2yrs",
    "fico_range_low", "fico_range_high", "fico_avg", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc", "application_type", "loan_status",
    "is_default", "default_rate_category", "income_bracket"
]
# Avoid duplicate columns if present
selected_cols = [c for c in selected_cols if c in df.columns]

df[selected_cols].to_csv(CLEAN_PATH, index=False)
print("Saved dashboard dataset to:", CLEAN_PATH)

In [ ]:
# Summary tables for Power BI usage
kpi_summary = pd.DataFrame({
    "Metric": ["Total Loans", "Default Rate", "Average Loan Amount", "Average Interest Rate"],
    "Value": [
        len(df),
        df["is_default"].mean() * 100,
        df["loan_amnt"].mean(),
        df["int_rate"].mean()
    ]
})

risk_by_grade = (
    df.groupby("grade")["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
)

risk_by_term = (
    df.groupby("term")["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
)

risk_by_purpose = (
    df.groupby("purpose")["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
)

risk_by_income = (
    df.groupby("income_bracket")["is_default"]
    .mean()
    .reset_index()
    .rename(columns={"is_default": "default_rate"})
)

risk_by_grade.to_csv(PROJECT_ROOT / "Dataset" / "dashboard_risk_by_grade.csv", index=False)
risk_by_term.to_csv(PROJECT_ROOT / "Dataset" / "dashboard_risk_by_term.csv", index=False)
risk_by_purpose.to_csv(PROJECT_ROOT / "Dataset" / "dashboard_risk_by_purpose.csv", index=False)
risk_by_income.to_csv(PROJECT_ROOT / "Dataset" / "dashboard_risk_by_income.csv", index=False)

print("Saved aggregate dashboard tables.")
print(kpi_summary)

In [ ]:
# Basic validation of created columns
print(df[["loan_status", "is_default", "grade", "loan_term_numeric", "fico_avg", "income_bracket"]].head())
print("Default rate overall:", df["is_default"].mean())